# Melanoma & Nevus Dermoscopy Dataset - Multimodal Training (Image + Metadata)

##### 1. Setup — chạy lại image pipeline

Giống `05_train_baseline.ipynb` — dùng `%run` để tái sử dụng `04_image_pipeline.ipynb`, tránh trùng lặp code. Trên Kaggle cần upload file này vào cùng Dataset đã "Add Input".

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = os.path.exists('/kaggle/input')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    pipeline_path = '/content/drive/MyDrive/Melanoma/04_image_pipeline.ipynb'
elif IN_KAGGLE:
    # Tìm file 04_image_pipeline.ipynb trong dataset đã "Add Input"
    pipeline_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if '04_image_pipeline.ipynb' in files:
            pipeline_path = os.path.join(root, '04_image_pipeline.ipynb')
            break
    if pipeline_path is None:
        raise FileNotFoundError(
            'Không tìm thấy 04_image_pipeline.ipynb trong /kaggle/input - '
            'nhớ upload file này vào dataset'
        )
else:
    pipeline_path = '04_image_pipeline.ipynb'

print('pipeline_path:', pipeline_path)
%run {pipeline_path}

Mounted at /content/drive
pipeline_path: /content/drive/MyDrive/Melanoma/04_image_pipeline.ipynb
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Giải nén /content/drive/MyDrive/Melanoma/images.zip ra /content/images ...
Xong.
IN_COLAB: True | IN_KAGGLE: True
IMAGES_DIR: /content/images
PROCESSED_DIR: /content/drive/MyDrive/Melanoma/data/processed
Train: (12696, 25) | Val: (2752, 25) | Test: (2685, 25)
Số file không tồn tại (trong 20 mẫu đầu): 0
Số batch - train: 397 | val: 86 | test: 84
images shape : torch.Size([32, 3, 224, 224])
labels shape : torch.Size([32]) | vài label đầu: [1, 0, 0, 0, 0, 1, 0, 1]
metadata shape: torch.Size([32, 13])


In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('So metadata feature:', len(METADATA_COLS))

Device: cuda
So metadata feature: 13


##### 2. Model — 2 nhánh: ảnh (ResNet18) + metadata (MLP nhỏ)

Nhánh ảnh: ResNet18 pretrained, bỏ layer `fc` cuối (dùng `nn.Identity()`), lấy thẳng feature vector 512 chiều.
Nhánh metadata: MLP nhỏ (13 → 32 chiều) cho `U50`/`U60`/`U70`/`sex_encoded`/`age_approx_scaled`/8 cột `anatom_*`.
Nối 2 nhánh (512+32=544 chiều) → 1 layer Linear ra output cuối (binary classification).

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights


class MultimodalModel(nn.Module):
    def __init__(self, n_metadata_features, image_feature_dim=512, metadata_hidden_dim=32):
        super().__init__()

        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        backbone.fc = nn.Identity()  # bỏ layer fc cuối, lấy thẳng feature 512 chiều
        self.image_branch = backbone

        self.metadata_branch = nn.Sequential(
            nn.Linear(n_metadata_features, metadata_hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Linear(image_feature_dim + metadata_hidden_dim, 1)

    def forward(self, image, metadata):
        image_features = self.image_branch(image)
        metadata_features = self.metadata_branch(metadata)
        combined = torch.cat([image_features, metadata_features], dim=1)
        return self.classifier(combined)


model = MultimodalModel(n_metadata_features=len(METADATA_COLS)).to(device)
print(model.classifier)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 213MB/s]


Linear(in_features=544, out_features=1, bias=True)


##### 3. Sanity check — forward pass thử 1 batch

In [ ]:
model.eval()
with torch.no_grad():
    sample_images, sample_labels, sample_metadata = next(iter(train_loader))
    sample_output = model(sample_images.to(device), sample_metadata.to(device))

print('Output shape:', sample_output.shape)  # kỳ vọng (batch, 1)

Output shape: torch.Size([32, 1])


##### 4. Loss, optimizer

Giống baseline — cùng `pos_weight` để bù mất cân bằng nhãn.

In [ ]:
import torch.optim as optim

n_nevus = (df_train['diagnosis'] == 'nevus').sum()
n_melanoma = (df_train['diagnosis'] == 'melanoma').sum()
pos_weight = torch.tensor([n_nevus / n_melanoma], device=device)
print('pos_weight:', pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

pos_weight: 1.506614017769003


##### 5. Hàm train 1 epoch / đánh giá

Giống baseline, chỉ khác: model nhận thêm `metadata` làm input.

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, accuracy_score, confusion_matrix


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, labels, metadata in loader:
        images = images.to(device)
        metadata = metadata.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images, metadata)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    for images, labels, metadata in loader:
        images = images.to(device)
        metadata = metadata.to(device)
        labels_t = labels.float().unsqueeze(1).to(device)

        outputs = model(images, metadata)
        loss = criterion(outputs, labels_t)
        total_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

    preds = [1 if p >= threshold else 0 for p in all_probs]

    metrics = {
        'loss': total_loss / len(loader.dataset),
        'accuracy': accuracy_score(all_labels, preds),
        'sensitivity': recall_score(all_labels, preds),  # recall cho lớp melanoma (1)
        'auc': roc_auc_score(all_labels, all_probs),
    }
    return metrics, all_labels, preds, all_probs

##### 6. Training loop với early stopping

In [ ]:
import os

EPOCHS = 15
PATIENCE = 3

if IN_COLAB:
    CHECKPOINT_PATH = os.path.join(DRIVE_ROOT, 'best_model_multimodal.pt')
elif IN_KAGGLE:
    CHECKPOINT_PATH = '/kaggle/working/best_model_multimodal.pt'
else:
    CHECKPOINT_PATH = 'best_model_multimodal.pt'

print('CHECKPOINT_PATH:', CHECKPOINT_PATH)

best_val_auc = 0.0
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics, _, _, _ = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_metrics['auc'])

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} "
          f"| val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
          f"val_sensitivity={val_metrics['sensitivity']:.4f} val_auc={val_metrics['auc']:.4f}")

    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f'  -> Lưu checkpoint mới (val_auc={best_val_auc:.4f})')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'Early stopping tại epoch {epoch} (không cải thiện sau {PATIENCE} epoch)')
            break

print('\nBest val AUC:', best_val_auc)

CHECKPOINT_PATH: /content/drive/MyDrive/Melanoma/best_model_multimodal.pt
Epoch 01 | train_loss=0.6271 | val_loss=0.5835 val_acc=0.7845 val_sensitivity=0.6488 val_auc=0.8523
  -> Lưu checkpoint mới (val_auc=0.8523)
Epoch 02 | train_loss=0.5564 | val_loss=0.5438 val_acc=0.7998 val_sensitivity=0.7412 val_auc=0.8675
  -> Lưu checkpoint mới (val_auc=0.8675)
Epoch 03 | train_loss=0.5104 | val_loss=0.5483 val_acc=0.7994 val_sensitivity=0.7468 val_auc=0.8740
  -> Lưu checkpoint mới (val_auc=0.8740)
Epoch 04 | train_loss=0.4782 | val_loss=0.5077 val_acc=0.8016 val_sensitivity=0.7810 val_auc=0.8851
  -> Lưu checkpoint mới (val_auc=0.8851)
Epoch 05 | train_loss=0.4467 | val_loss=0.5836 val_acc=0.8049 val_sensitivity=0.6442 val_auc=0.8676
Epoch 06 | train_loss=0.4218 | val_loss=0.5407 val_acc=0.7834 val_sensitivity=0.8364 val_auc=0.8792
Epoch 07 | train_loss=0.3574 | val_loss=0.5127 val_acc=0.8172 val_sensitivity=0.7671 val_auc=0.8983
  -> Lưu checkpoint mới (val_auc=0.8983)
Epoch 08 | train_loss

##### 7. Đánh giá cuối cùng trên test set (chỉ chạy 1 lần, threshold=0.5)

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

test_metrics, test_labels, test_preds, test_probs = evaluate(model, test_loader, criterion, device)

print('=== Test set metrics (threshold=0.5) ===')
for k, v in test_metrics.items():
    print(f'{k}: {v:.4f}')

print('\nConfusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):')
print(confusion_matrix(test_labels, test_preds))

=== Test set metrics (threshold=0.5) ===
loss: 0.5749
accuracy: 0.8071
sensitivity: 0.7905
auc: 0.8784

Confusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):
[[1333  297]
 [ 221  834]]


##### 8. Tune lại threshold trên val set (Youden's J) — giống baseline, để so sánh công bằng

In [ ]:
import numpy as np

val_metrics, val_labels, _, val_probs = evaluate(model, val_loader, criterion, device)

thresholds = np.arange(0.01, 1.0, 0.01)
best_threshold = 0.5
best_j = -1.0

for t in thresholds:
    preds = [1 if p >= t else 0 for p in val_probs]
    tn, fp, fn, tp = confusion_matrix(val_labels, preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    j = sensitivity + specificity - 1

    if j > best_j:
        best_j = j
        best_threshold = t

print(f"Threshold mặc định: 0.50")
print(f"Threshold tối ưu (Youden's J trên val): {best_threshold:.2f} (J={best_j:.4f})")

Threshold mặc định: 0.50
Threshold tối ưu (Youden's J trên val): 0.46 (J=0.6283)


In [ ]:
test_metrics_tuned, test_labels_tuned, test_preds_tuned, _ = evaluate(
    model, test_loader, criterion, device, threshold=best_threshold
)

print(f'=== Test set metrics (threshold={best_threshold:.2f}, tuned trên val) ===')
for k, v in test_metrics_tuned.items():
    print(f'{k}: {v:.4f}')

print('\nConfusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):')
print(confusion_matrix(test_labels_tuned, test_preds_tuned))

=== Test set metrics (threshold=0.46, tuned trên val) ===
loss: 0.5749
accuracy: 0.8015
sensitivity: 0.8076
auc: 0.8784

Confusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):
[[1300  330]
 [ 203  852]]
